# Chapter 5 — Inference Engines

**The Encyclopedia of Artificial Intelligence** — Perla Genovesi

This notebook contains the Python example from Chapter 5. Press the play button (▶) next to the code cell to run it: the result appears just below.

No installation is needed. The code uses only Python's standard library.

In [ ]:
from collections import deque

# Horn clauses: (name, premises, conclusion)
RULES = [
    ("R1", ["valid_address", "product_available"], "order_fulfillable"),
    ("R2", ["payment_authorised", "order_fulfillable"],
     "shipment_scheduled"),
    ("R3", ["valid_card", "sufficient_funds"], "payment_authorised"),
    ("R4", ["in_stock"], "product_available"),
    ("R5", ["arriving_within_2_days"], "product_available"),
    ("R6", ["loyal_customer"], "free_coupon"),
]

FACTS = ["valid_address", "in_stock", "valid_card",
         "sufficient_funds", "loyal_customer"]

def forward(rules, facts):
    """Forward chaining with premise counters (linear time)."""
    missing = {name: len(premises) for name, premises, _ in rules}
    uses = {}                                  # fact -> rules that use it
    for name, premises, conclusion in rules:
        for p in premises:
            uses.setdefault(p, []).append((name, conclusion))
    known, queue, steps = set(), deque(facts), []
    while queue:
        fact = queue.popleft()
        if fact in known:
            continue
        known.add(fact)
        for name, conclusion in uses.get(fact, []):
            missing[name] -= 1
            if missing[name] == 0:             # all premises are true
                steps.append(f"{name} -> {conclusion}")
                queue.append(conclusion)
    return known, steps

def backward(goal, rules, facts, level=0, in_progress=frozenset()):
    """Backward chaining: prints the proof of the goal."""
    indent = "  " * level
    if goal in facts:
        print(f"{indent}{goal}: known fact")
        return True
    if goal in in_progress:                    # avoid loops
        return False
    for name, premises, conclusion in rules:
        if conclusion == goal:
            print(f"{indent}{goal}: trying {name}")
            if all(backward(p, rules, facts, level + 1,
                            in_progress | {goal})
                   for p in premises):
                print(f"{indent}{goal}: proved with {name}")
                return True
    print(f"{indent}{goal}: cannot be proved")
    return False

known, steps = forward(RULES, FACTS)
print("Forward:", ", ".join(steps))
print("shipment_scheduled derived:", "shipment_scheduled" in known)
print()
backward("shipment_scheduled", RULES, set(FACTS))

## Expected output

If everything works, you should see:

```
Forward: R4 -> product_available, R3 -> payment_authorised,
         R6 -> free_coupon, R1 -> order_fulfillable,
         R2 -> shipment_scheduled
shipment_scheduled derived: True
```

Now try changing a number in the code and run it again to see what happens.